In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, types # Export to DataBase

# read XPT file 
df_diabetes = pd.read_sas("../data/questionaire_data/DIQ_L.xpt", format="xport", encoding="utf-8")
df_diabetes

,SEQN,DIQ010,DID040,DIQ160,DIQ180,DIQ050,DID060,DIQ060U,DIQ070
0,130378.0,2.0,NaN,2.0,2.0,NaN,NaN,NaN,NaN
1,130379.0,2.0,NaN,2.0,1.0,NaN,NaN,NaN,NaN
2,130380.0,1.0,35.0,NaN,NaN,2.0,NaN,NaN,1.0
3,130381.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,130382.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
11739,142306.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11740,142307.0,1.0,42.0,NaN,NaN,2.0,NaN,NaN,1.0
11741,142308.0,2.0,NaN,2.0,2.0,NaN,NaN,NaN,NaN
11742,142309.0,2.0,NaN,2.0,2.0,NaN,NaN,NaN,NaN


In [ ]:
print("Diabetes Questionaire Data Info:")
df_diabetes.info() 

# SEQN - Respondent sequence number
# DIQ010 - Doctor told you have diabetes
# DID040 - Age when first told you had diabetes
# DIQ159 - CHECK ITEM
# DIQ160 - Ever told you have prediabetes
# DIQ180 - Had blood tested past three years
# DIQ050 - Taking insulin now
# DID060 - How long taking insulin
# DIQ060U - Unit of measure (month/year)
# DIQ065 - CHECK ITEM
# DIQ070 - Take diabetic pills to lower blood sugar

Diabetes Questionaire Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11744 entries, 0 to 11743
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   SEQN     11744 non-null  float64
 1   DIQ010   11740 non-null  float64
 2   DID040   1081 non-null   float64
 3   DIQ160   8022 non-null   float64
 4   DIQ180   8304 non-null   float64
 5   DIQ050   1081 non-null   float64
 6   DID060   343 non-null    float64
 7   DIQ060U  332 non-null    float64
 8   DIQ070   2281 non-null   float64
dtypes: float64(9)
memory usage: 825.9 KB


In [5]:
# Define the columns you want to keep and their new names
columns_to_keep_diab = {
    'SEQN':   'Participant_ID',
    'DIQ010': 'Diagnosed_Diabetes',
    'DIQ160': 'Diagnosed_Prediabetes',
    'DIQ050': 'Currently_Insulin_Therapy',
    'DIQ070': 'Currently_Diabetic_Meds'
}

# Select only the desired columns from the original DataFrame
df_diabetes_selected = df_diabetes[list(columns_to_keep_diab.keys())].copy()

# Rename the columns in the new DataFrame
df_diabetes_selected.rename(columns=columns_to_keep_diab, inplace=True)


# --- Verification Steps ---
print("--- Selected and Renamed Diabetes Data Info ---")
df_diabetes_selected.info()

print("\n--- First few rows of the new DataFrame ---")
print(df_diabetes_selected.head())

--- Selected and Renamed Diabetes Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11744 entries, 0 to 11743
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Participant_ID             11744 non-null  float64
 1   Diagnosed_Diabetes         11740 non-null  float64
 2   Diagnosed_Prediabetes      8022 non-null   float64
 3   Currently_Insulin_Therapy  1081 non-null   float64
 4   Currently_Diabetic_Meds    2281 non-null   float64
dtypes: float64(5)
memory usage: 458.9 KB

--- First few rows of the new DataFrame ---
   Participant_ID  Diagnosed_Diabetes  Diagnosed_Prediabetes  \
0        130378.0                 2.0                    2.0   
1        130379.0                 2.0                    2.0   
2        130380.0                 1.0                    NaN   
3        130381.0                 2.0                    NaN   
4        130382.0                 2.0        

In [6]:
df_diabetes_selected = df_diabetes_selected.convert_dtypes() 
df_diabetes_selected

,Participant_ID,Diagnosed_Diabetes,Diagnosed_Prediabetes,Currently_Insulin_Therapy,Currently_Diabetic_Meds
0,130378,2,2,<NA>,<NA>
1,130379,2,2,<NA>,<NA>
2,130380,1,<NA>,2,1
3,130381,2,<NA>,<NA>,<NA>
4,130382,2,<NA>,<NA>,<NA>
...,...,...,...,...,...
11739,142306,2,<NA>,<NA>,<NA>
11740,142307,1,<NA>,2,1
11741,142308,2,2,<NA>,<NA>
11742,142309,2,2,<NA>,<NA>


In [9]:
# export as csv
file_path = "../data/questionaire_data/cleaned_diabetes.csv" 

try:
    df_diabetes_selected.to_csv(file_path, index=False, encoding='utf-8')
    print(f"DataFrame successfully saved to: {file_path}")
except Exception as e:
    print(f"Error saving DataFrame to CSV: {e}")

DataFrame successfully saved to: ../data/questionaire_data/cleaned_diabetes.csv


In [8]:
from dotenv import dotenv_values

config = dotenv_values()

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

# Now building the URL with the values from the .env file
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

engine = create_engine(url, echo=False) 

In [10]:
df_diabetes_selected.to_sql(name = 'diabetes_questionaire_data', con=engine, schema='capstone_group_3',if_exists='replace',index=False)

744